<a href="https://colab.research.google.com/github/AniShriyak/Bio-Inspired-Lab/blob/main/BIS_AntColoney.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

def ant_colony_tsp(
    distance,
    n_ants=10,
    n_iterations=100,
    alpha=1.0,
    beta=2.0,
    evaporation=0.5,
    Q=100
):
    n = len(distance)

    # Step 1: Initialize pheromone levels
    pheromone = np.ones((n, n))
    np.fill_diagonal(pheromone, 0)

    # Short distances are more attractive
    heuristic = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j and distance[i][j] > 0:
                heuristic[i][j] = 1 / distance[i][j]

    best_tour = None
    best_length = float("inf")

    # Step 2: Repeat the search
    for iteration in range(n_iterations):
        all_tours = []
        all_lengths = []

        # Step 3: Each ant builds a tour
        for ant in range(n_ants):
            start = np.random.randint(n)
            tour = [start]
            unvisited = set(range(n))
            unvisited.remove(start)

            while unvisited:
                current = tour[-1]
                cities = list(unvisited)

                weights = np.array([
                    (pheromone[current][city] ** alpha)
                    * (heuristic[current][city] ** beta)
                    for city in cities
                ])

                probabilities = weights / weights.sum()
                next_city = np.random.choice(
                    cities, p=probabilities
                )

                tour.append(next_city)
                unvisited.remove(next_city)

            # Return to the starting city
            tour.append(start)

            # Calculate total tour distance
            length = sum(
                distance[tour[i]][tour[i + 1]]
                for i in range(len(tour) - 1)
            )

            all_tours.append(tour)
            all_lengths.append(length)

            # Save the best tour found so far
            if length < best_length:
                best_length = length
                best_tour = tour.copy()

        # Step 4: Evaporate old pheromones
        pheromone *= (1 - evaporation)

        # Step 5: Deposit new pheromones
        for tour, length in zip(all_tours, all_lengths):
            deposit = Q / length

            for i in range(len(tour) - 1):
                a = tour[i]
                b = tour[i + 1]

                pheromone[a][b] += deposit
                pheromone[b][a] += deposit

    return best_tour, best_length


# Example: 5 cities and their distances
distance = np.array([
    [0,  2,  9, 10,  7],
    [2,  0,  6,  4,  3],
    [9,  6,  0,  8,  5],
    [10, 4,  8,  0,  6],
    [7,  3,  5,  6,  0]
], dtype=float)

best_tour, best_length = ant_colony_tsp(distance)

print("Best tour:", " -> ".join(map(str, best_tour)))
print("Shortest distance found:", best_length)


Best tour: 2 -> 0 -> 1 -> 3 -> 4 -> 2
Shortest distance found: 26.0
